# Module 1.2: Preventive Laptop Maintenance, Thermal Refurbishment, Internal NVMe Migration & BIOS Server Tuning

## Homelab Expansion (Chapter 2) - Step 1: Hardware Refurbishment & Battery Bypass

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Safely export your Chapter 1 Immich PostgreSQL database (`pg_dumpall`) and `.env` onto the WD Blue SN5000 NVMe SSD, deactivate the Pi's SSD swapfile, and cleanly unmount `/mnt/homelab_storage` before physically extracting the drive.
2. Remove the shared copper heatpipe assembly using the factory reverse-numerical screw sequence, clean internal blower fin carpets, and repaste the bare CPU and NVIDIA GPU dies with high-viscosity thermal compound (`Arctic MX-6`, `Noctua NT-H2`, or `PTM7950`) plus viscous `K5 PRO` putty on VRMs/VRAM.
3. Install your Chapter 1 **WD Blue SN5000 NVMe SSD** directly into the 2019 ROG Zephyrus's open **second internal M.2 2280 NVMe slot**—eliminating USB bridge bottlenecks and gaining native PCIe NVMe reliability and SMART telemetry.
4. Configure the ASUS ROG UEFI BIOS for 24/7 headless Linux server operation (`Secure Boot` disabled, `Fast Boot` disabled, `AHCI` enabled) and verify that **both** internal M.2 NVMe drives are detected before closing the hardware phase.

## 1.2.1 Conceptual Background: Thermal Physics & Why Internal M.2 NVMe Beats a USB Enclosure

A 2019 gaming laptop has experienced roughly 6–7 years of thermal expansion and contraction cycles. Because we already have the bottom chassis open during Step 1 maintenance, this is also the cleanest single-open opportunity to migrate your **WD Blue SN5000 NVMe SSD** from the Raspberry Pi's USB 3.0 enclosure directly into the ROG Zephyrus's open **second M.2 2280 slot**—preparing `homelab-gpu` to become your unified Immich + App/AI server while freeing `homelab-pi` to act strictly as a dedicated **Tier-0 Network & Monitoring box**.

### 1. The "Pump-Out" Effect on Bare Laptop Silicon Dies
Unlike desktop CPUs that have an Integrated Heat Spreader (IHS), laptop CPUs and NVIDIA GPUs expose the bare mirror-finish silicon die directly to the copper heatsink.
- **Why Thin Pastes Fail:** Low-viscosity desktop pastes (like standard Arctic MX-4 or Kryonaut) thin out above 80°C and gradually squeeze out from between the silicon die and copper plate ("pump-out"), leaving dry hotspots within months.
- **The Solution:** Use either a **Honeywell PTM7950 phase-change pad** (which stays solid below 45°C and liquefies microscopically under load) or a **high-viscosity non-conductive paste** (**Arctic MX-6**, **Noctua NT-H2**, or **Thermalright TFX**).
- *Note on GX502 Liquid Metal Variants:* Standard `GA502` and `GU502` models use conventional grey thermal paste. If you have a high-end `GX502` factory-equipped with liquid metal on the CPU (identifiable by a white foam barrier glued around the CPU socket and mirror-like gallium alloy), clean and refresh thoughtfully or replace carefully after removing all traces of gallium with 99% IPA.

### 2. Why `K5 PRO` Viscous Thermal Putty Beats Solid Pads on ROG Zephyrus
On the 2019 ROG Zephyrus motherboard, the copper heatsink cools not only the CPU and GPU dies, but also **6–8 GDDR6 VRAM modules** and **over a dozen VRM MOSFETs/chokes** surrounding the sockets.
- **The Rigid Pad Trap:** ASUS engineered the Zephyrus heatsink with ultra-tight, non-uniform clearances (ranging from `0.3mm` to `0.8mm`) and originally applied factory thermal putty on the VRMs. If you replace that putty with standard `1.0mm` solid silicone thermal pads, the pads act as stilts—holding the copper coldplate `0.2mm` off the CPU and GPU dies and causing immediate 100°C thermal shutdowns!
- **Viscous Putty (`K5 PRO` / `Upsiren U6 Pro`):** Putty has the consistency of thick frosting. When you tighten the numbered heatsink screws, it squishes down to the exact microscopic gap (`0.1mm` to `2.0mm`) without pushing back against the CPU/GPU mounting pressure.

### 3. Why Moving the WD Blue SN5000 Into Internal M.2 Slot 2 Is a Major Upgrade
Every 2019 ASUS ROG Zephyrus (`GA502`, `GU502`, and `GX502`) features **two internal M.2 2280 NVMe slots** on the motherboard:
- **M.2 Slot 1 (Stock OS SSD):** Will be wiped in Module 2.1 to host **Ubuntu Server 24.04 LTS**.
- **M.2 Slot 2 (Open Expansion Slot):** Accepts your single-sided PCIe Gen4 **WD Blue SN5000** directly (backward-compatible with PCIe Gen3 x2 ~1,700 MB/s on `GA502` or Gen3 x4 ~3,500 MB/s on `GU502`/`GX502`).
- **Why Native PCIe Beats USB 3.0 UASP:** You eliminate the Realtek USB-to-NVMe bridge chip, gain 3×–6× lower I/O latency for PostgreSQL (`Immich`, `Paperless-ngx`, `Mail-Archiver`), unlock native `nvme smart-log` temperature/wear telemetry, and eliminate accidental USB cable disconnects—while freeing your UGREEN enclosure as a spare diagnostic tool!
- **Why We Export PostgreSQL with `pg_dumpall` *Before* Moving the Drive:** PostgreSQL's raw on-disk binary data directory (`/mnt/homelab_storage/docker_volumes/immich_postgres`) is **CPU-architecture specific**—files written by 64-bit ARM (`aarch64`) on `homelab-pi` cannot be mounted directly by 64-bit x86 (`x86_64`) on `homelab-gpu`. Saving a portable logical SQL dump (`immich_migration_dump.sql.gz`) right onto `/mnt/homelab_storage/` before stopping Immich guarantees a 100% clean cross-architecture database restore in Module 4.2!

### 4. Essential Preventive Hardware Checks During Teardown
While the laptop is open on your bench, inspect these four often-overlooked failure points:
1. **Blower Fan Fin Carpets:** Dust fibers mat into a solid felt wall *between* the blower fan wheel and the inner side of the copper radiator fins (invisible from the outside).
2. **M.2 Slot 2 Standoff & `M2×3mm` Mounting Screw:** Check whether the empty M.2 2280 standoff already has a small mounting screw pre-threaded into it (or use a spare `M2×3mm` screw) and ensure Wi-Fi antenna coax snaps (`MAIN`/`AUX`) are firmly seated.
3. **Coin-Cell CMOS Battery (if present):** Because the main Li-Po battery is now removed, the motherboard relies on its small `CR2032` / 2-pin wired RTC coin cell (on models that include one) to retain BIOS settings when unplugged from the UPS.
4. **ErgoLift Hinge Screws:** Check that the 2–3 structural screws anchoring the left and right display hinges to the chassis frame have not backed out over years of opening and closing.

## 1.2.2 Step-by-Step Laboratory Guide

### Task 1.2.1: Pre-Flight on `homelab-pi` — Export Immich DB & Safely Unmount the WD Blue SN5000
> ⚠️ **CRITICAL DATA-SAFETY RULE:** Never unplug a mounted Linux `ext4` drive while PostgreSQL and an active `swapfile` are running on it! Perform this 60-second pre-flight on `homelab-pi` *before* touching the USB cable.

1. SSH into your Raspberry Pi (`ssh homelab-pi`) and create a compressed, cross-architecture logical PostgreSQL dump plus a backup copy of your Chapter 1 `.env` directly on the NVMe SSD:
   ```bash
   # 1. Export the entire Immich PostgreSQL database logically (ARM64 -> x86_64 portable!)
   docker exec -t immich_postgres pg_dumpall --clean --if-exists --username=postgres \
     | gzip > /mnt/homelab_storage/immich_migration_dump.sql.gz

   # 2. Verify the dump is non-empty and valid gzip
   ls -lh /mnt/homelab_storage/immich_migration_dump.sql.gz
   gzip -t /mnt/homelab_storage/immich_migration_dump.sql.gz && echo "SQL dump integrity OK!"

   # 3. Copy your Chapter 1 Immich .env (preserves DB_PASSWORD and settings) onto the SSD
   cp ~/homelab/stacks/immich/.env /mnt/homelab_storage/immich_migration_env.bak
   chmod 600 /mnt/homelab_storage/immich_migration_env.bak
   ```
2. Stop the Immich containers cleanly, deactivate the Chapter 1 NVMe swapfile, flush write buffers, and unmount `/mnt/homelab_storage`:
   ```bash
   # 4. Gracefully stop Immich so all disk buffers flush cleanly
   cd ~/homelab/stacks/immich
   docker compose down

   # 5. Deactivate the 2GB SSD swapfile from Chapter 1, Module 5.1
   sudo swapoff /mnt/homelab_storage/swapfile
   swapon --show

   # 6. Sync caches and unmount the WD Blue SN5000 cleanly
   sync
   sudo umount /mnt/homelab_storage
   mountpoint /mnt/homelab_storage || echo "SSD safely unmounted! Ready to unplug."
   ```
   *(Note: We will finish stripping the stale `/etc/fstab` entries and old Immich containers off `homelab-pi` in Module 2.2 right after `homelab-gpu` is powered on and verified!)*
3. Unplug the UGREEN USB-C NVMe enclosure from `homelab-pi`, slide open the enclosure, remove the rubber fastener/screw, and carefully extract the bare **WD Blue SN5000 M.2 2280 SSD** onto an anti-static surface.

### Task 1.2.2: Heatsink Removal, Deep Fin Cleaning, `K5 PRO` Putty & CPU/GPU Repaste
1. Confirm the ROG Zephyrus is **unplugged from AC power** (battery already removed in Module 1.1) and remove the bottom cover.
2. **Disconnect the Dual Cooling Fans:** Gently unclip the 4-pin PWM fan cables for both the CPU fan and GPU fan from their motherboard headers.
3. **Unscrew Heatsink in Reverse Numerical Order (`8 → 1`):** Locate the stamped numbers (`1` through `7` or `8`) around the CPU/GPU brackets and loosen each captive spring screw in **reverse numerical order**, backing each out a couple of turns first before fully releasing.
4. **Lift & Clean Blower Fans + Copper Fins:** Gently lift the copper heatpipe assembly straight up, brush out the internal "dust carpets" trapped between the blower wheels and exhaust fins, and **always hold fan blades stationary** if using compressed air.
5. **Clean Silicon Dies with 99% IPA:** Scrub all baked-on factory paste from the mirror-finish CPU and NVIDIA GPU dies and the underside copper coldplates, and clear away dried/crumbly factory VRM putty.
6. **Apply Viscous Putty (`K5 PRO`) & High-Viscosity Die Paste:**
   - Dab pea-sized beads of `K5 PRO` / `Upsiren U6 Pro` onto every GDDR6 VRAM chip and VRM MOSFET row.
   - Apply a thin, **100%-coverage** buttered layer of `Arctic MX-6` / `Noctua NT-H2` (or exact-cut `PTM7950` pad) across the bare rectangular CPU and NVIDIA GPU dies.
7. **Reinstall Heatsink in Forward Numerical Order (`1 → 8`):** Seat the heatsink straight down, thread screws 2 turns each in order, then torque fully in order (`1 → N`). **Reconnect both PWM fan cables!**

### Task 1.2.3: Install the WD Blue SN5000 Into Internal M.2 Slot 2
Before reinstalling the laptop's bottom cover:
1. Locate the **open second M.2 2280 NVMe slot** on the ROG Zephyrus motherboard (note which slot already holds the factory Windows OS drive — leave the factory drive in Slot 1 for Ubuntu Server!).
2. Remove the small **`M2×3mm` mounting screw** from the Slot 2 brass standoff (if pre-installed; otherwise use a spare `M2×3mm` NVMe screw).
3. Insert the **WD Blue SN5000** into the M.2 M-key connector at a gentle **~20° angle** until the gold contacts seat completely.
4. Press the tail of the SSD flat onto the standoff and tighten the `M2×3mm` screw snugly (do not over-torque). If your laptop bottom plate includes a thermal pad aligned over Slot 2, verify its protective film is peeled.
5. Reinstall the bottom chassis cover using your screw map from Module 1.1.

### 🖼️ Task 1.2.3 Verification: Repasted Cooling System & Dual Internal M.2 NVMe Drives

**Verification Objective:** Capture a photo of the open ROG Zephyrus interior showing the cleaned/repasted cooling assembly and your **WD Blue SN5000 NVMe SSD** seated inside internal M.2 Slot 2 alongside the stock OS NVMe drive.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.2.4: ASUS ROG UEFI BIOS Configuration & Dual-NVMe Detection Check
With the chassis reassembled and plugged into your UPS-protected AC adapter:
1. Power on the laptop and hold `F2` to enter the **ASUS ROG UEFI BIOS Utility**, then press `F7` to switch to **Advanced Mode**.
2. Navigate to **Advanced $\rightarrow$ NVMe Configuration** (or check the Main/Storage summary pane) and verify **both internal NVMe drives** are detected:
   - **Drive 1:** Stock factory SSD *(will be wiped for Ubuntu Server 24.04 LTS in Module 2.1)*.
   - **Drive 2:** **`WD Blue SN5000`** *(holds your intact Chapter 1 Immich photo library + `immich_migration_dump.sql.gz` — DO NOT format during OS install!)*.
   - *Write down the exact model name and capacity of each drive so you cannot mix them up in Module 2.1!*
3. Configure the following settings for headless Ubuntu Server and proprietary NVIDIA driver compatibility:
   - **Security $\rightarrow$ Secure Boot $\rightarrow$ Secure Boot Control:** Set to **`Disabled`**. *(Allows Ubuntu's proprietary NVIDIA kernel modules `nvidia.ko` and `nvidia-uvm.ko` to load cleanly across kernel upgrades without Manual Machine Owner Key [MOK] enrollment on a headless machine).*
   - **Boot $\rightarrow$ Fast Boot:** Set to **`Disabled`**. *(Ensures full PCIe/NVMe and USB bus initialization on every cold boot).*
   - **Advanced $\rightarrow$ SVM Mode (AMD Ryzen GA502) or Intel Virtualization Technology / VT-x (Intel GU502/GX502):** Set to **`Enabled`** for Docker/KVM support.
   - **Advanced $\rightarrow$ SATA Mode Selection:** Ensure set to **`AHCI`** (never Intel RST RAID) so Linux sees both NVMe drives natively as `/dev/nvme0n1` and `/dev/nvme1n1`.
4. Press `F10` to **Save Changes and Reset**.

### 🖼️ Task 1.2.4 Verification: ASUS UEFI BIOS Server Configuration & Dual-NVMe Detection

**Verification Objective:** Capture a photo of the ASUS UEFI BIOS Advanced Mode screen showing `Secure Boot Control: Disabled`, virtualization enabled, and both NVMe drives (stock SSD + WD Blue SN5000) recognized.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Logical Immich database dump (`immich_migration_dump.sql.gz`) and `.env` backup saved onto `/mnt/homelab_storage/` on `homelab-pi`, Immich stopped, swapfile deactivated, and SSD cleanly unmounted.
- [ ] Blower fans/radiators cleaned, bare CPU & GPU dies repasted, VRM/VRAM coated with `K5 PRO` putty, heatsink torqued (`1 → N`), and **both fan cables reconnected**.
- [ ] **WD Blue SN5000** installed cleanly into the ROG Zephyrus's open **internal M.2 Slot 2**.
- [ ] UEFI BIOS configured (`Secure Boot: Disabled`, `Fast Boot: Disabled`, `AHCI`) and **both** NVMe drives detected.
- [ ] Verification photos captured and attached above.

Proceed to [Module 2.1: Replacing Unlicensed Windows 11 with Ubuntu Server 24.04 LTS & Mounting Internal NVMe Storage](../02_os_installation_and_headless_setup/02_1_os_selection_and_ubuntu_install.ipynb).